# Single-UAV Mission Planning with a Genetic Algorithm
**Artificial Intelligence – BSCS 5 · Assignment 2 · Bahria University Lahore Campus**

**How to run:** `Runtime → Run all`. Nothing needs to be installed (Colab already has numpy and matplotlib).

Each code cell below is one file of the GitHub repository (same code), so you can explain the project
cell by cell. All figures and CSV tables are saved in the `results` folder (left side → Files).

## 1. Settings (`config.py`)

In [ ]:
"""
All settings of the project (plain variables).
Change a number here and run the notebook again (Runtime -> Run all).
"""

# ---------------- The map (a 50 x 50 grid, 1 cell = 20 m) ----------------
GRID_SIZE = 50
MAP_SEED = 7               # fixed seed -> the targets are always in the same place

# No-fly zones as rectangles (x1, y1, x2, y2): cells x1..x2-1, y1..y2-1 are blocked
NO_FLY_ZONES = [(12, 8, 18, 30), (28, 0, 34, 14), (24, 30, 36, 36)]

# Hazard zones (high wind / radiation) as circles (centre_x, centre_y, radius)
HAZARD_ZONES = [(42, 24, 8), (22, 20, 7), (10, 40, 6)]

# Disaster (priority) zones as rectangles (x1, y1, x2, y2, importance 0..1)
PRIORITY_ZONES = [(36, 18, 48, 30, 1.0), (4, 34, 16, 46, 0.8), (20, 40, 30, 48, 0.6)]
BACKGROUND_PRIORITY = 0.1  # every other cell is a little bit useful too
N_TARGETS = 15             # targets placed inside the priority zones

# ---------------- The drone ----------------
N_DRONES = 1               # a single drone surveys the area
N_WAYPOINTS = 6            # waypoints of the drone (it also has a deployment position)
SPEED_MIN, SPEED_MAX = 1.0, 3.0   # cells per time step
MAX_RANGE = 150            # max path length of the drone (cells), including the way back

# ---------------- Fitness weights ----------------
W_COVERAGE = 0.35
W_DETECTION = 0.25
W_ENERGY = 0.15
W_TIME = 0.10
W_HAZARD = 0.15
PENALTY_WEIGHT = 2.0       # how hard constraint violations are punished

# ---------------- Genetic Algorithm ----------------
SEED = 42
POP_SIZE = 50
GENERATIONS = 100
CROSSOVER_RATE = 0.9
MUTATION_RATE = 0.1        # chance that one gene group (waypoint / deployment / sensor bit) is mutated
MUTATION_STEP = 3.0        # how far (cells) a mutated waypoint moves on average
TOURNAMENT_K = 3           # tournament size = selection pressure
ELITE = 2                  # best individuals copied unchanged to the next generation
IMMIGRANTS = 2             # new random individuals per generation (keeps diversity)
PATIENCE = 30              # stop if the best fitness did not improve for 30 generations

# notebook only: collect all settings in one object called cfg (like `import config as cfg`)
import types
cfg = types.SimpleNamespace(**{k: v for k, v in dict(globals()).items() if k.isupper()})

## 2. The grid world (`environment.py`)

In [ ]:
"""
The grid world: no-fly zones, hazard zones, priority (disaster) zones and targets.
Arrays are indexed as grid[x, y].
"""
import numpy as np



class Environment:
    def __init__(self):
        n = cfg.GRID_SIZE
        self.size = n

        # 1 = no-fly (blocked), 0 = open
        self.no_fly = np.zeros((n, n), dtype=bool)
        for x1, y1, x2, y2 in cfg.NO_FLY_ZONES:
            self.no_fly[x1:x2, y1:y2] = True

        # centre coordinates of every cell (used to check which cells a sensor sees)
        xs, ys = np.meshgrid(np.arange(n) + 0.5, np.arange(n) + 0.5, indexing="ij")
        self.cell_x, self.cell_y = xs.ravel().astype(np.float32), ys.ravel().astype(np.float32)

        # hazard: 1 in the middle of a hazard zone, falling to 0 at its edge
        self.hazard = np.zeros((n, n))
        for cx, cy, r in cfg.HAZARD_ZONES:
            dist = np.hypot(xs - cx, ys - cy)
            self.hazard = np.maximum(self.hazard, np.clip(1 - dist / r, 0, 1))

        # priority: how important it is to survey each cell
        self.priority = np.full((n, n), cfg.BACKGROUND_PRIORITY)
        for x1, y1, x2, y2, weight in cfg.PRIORITY_ZONES:
            self.priority[x1:x2, y1:y2] = weight
        self.priority[self.no_fly] = 0.0

        # targets: random open cells inside the priority zones (same every run)
        rng = np.random.default_rng(cfg.MAP_SEED)
        cells = np.argwhere((self.priority >= 0.5) & ~self.no_fly)
        chosen = rng.choice(len(cells), cfg.N_TARGETS, replace=False)
        self.targets = cells[chosen] + 0.5

    def is_blocked(self, x, y):
        """True if the point (x, y) is outside the map or inside a no-fly zone."""
        if not (0 <= x < self.size and 0 <= y < self.size):
            return True
        return bool(self.no_fly[int(x), int(y)])

    def nearest_free(self, x, y):
        """Closest open cell centre to (x, y): search in growing squares around it."""
        cx, cy = int(np.clip(x, 0, self.size - 1)), int(np.clip(y, 0, self.size - 1))
        for r in range(1, self.size):
            for i in range(cx - r, cx + r + 1):
                for j in range(cy - r, cy + r + 1):
                    if not self.is_blocked(i, j):
                        return i + 0.5, j + 0.5
        return x, y

## 3. The chromosome (`chromosome.py`)

In [ ]:
"""
The chromosome = the complete plan for the drone (a hybrid real + binary encoding):

    [ deploy_x, deploy_y | (x, y, speed, power, on) x N_WAYPOINTS ]

    deploy_x, deploy_y   deployment position: where the drone is launched and lands again
    x, y                 waypoint position (cells)
    speed                speed on the way to this waypoint (SPEED_MIN..SPEED_MAX)
    power                sensor power 0..1 (stronger sensor = bigger circle, more energy)
    on                   1 = sensor ON on this leg, 0 = OFF  (the sensor activation schedule)

The drone flies: deployment position -> waypoint 1 -> ... -> waypoint 6 -> deployment position
"""
import numpy as np



class Chromosome:
    def __init__(self, deploy, genes):
        self.deploy = deploy        # array (N_DRONES = 1, 2): deployment position
        self.genes = genes          # array (N_DRONES = 1, N_WAYPOINTS, 5): x, y, speed, power, on
        self.fitness = None         # filled in by the fitness function
        self.details = None         # the separate objective values

    @staticmethod
    def random(rng):
        """A random plan: random deployment position, waypoints, speeds, powers and on/off bits."""
        shape = (cfg.N_DRONES, cfg.N_WAYPOINTS)
        deploy = rng.uniform(0, cfg.GRID_SIZE, (cfg.N_DRONES, 2))
        genes = np.stack([
            rng.uniform(0, cfg.GRID_SIZE, shape),              # x
            rng.uniform(0, cfg.GRID_SIZE, shape),              # y
            rng.uniform(cfg.SPEED_MIN, cfg.SPEED_MAX, shape),  # speed
            rng.uniform(0, 1, shape),                          # sensor power
            rng.integers(0, 2, shape),                         # sensor on (1) / off (0)
        ], axis=2).astype(float)
        return Chromosome(deploy, genes)

    def copy(self):
        new = Chromosome(self.deploy.copy(), self.genes.copy())
        new.fitness, new.details = self.fitness, self.details
        return new

    def route(self, drone):
        """All points the drone visits: deployment position -> waypoints -> deployment position."""
        start = self.deploy[drone]
        return np.vstack([start, self.genes[drone, :, :2], start])

    def clip(self):
        """Keep every gene inside its allowed range."""
        g = self.genes
        self.deploy[:] = np.clip(self.deploy, 0, cfg.GRID_SIZE - 0.01)
        g[:, :, 0:2] = np.clip(g[:, :, 0:2], 0, cfg.GRID_SIZE - 0.01)
        g[:, :, 2] = np.clip(g[:, :, 2], cfg.SPEED_MIN, cfg.SPEED_MAX)
        g[:, :, 3] = np.clip(g[:, :, 3], 0, 1)
        g[:, :, 4] = np.round(np.clip(g[:, :, 4], 0, 1))      # the on/off gene is always 0 or 1

## 4. Fitness function and constraint handling (`fitness.py`)

In [ ]:
"""
Fitness function and constraint handling.

    F = 0.35*Coverage + 0.25*Detection - 0.15*Energy - 0.10*Time - 0.15*Hazard - 2*Penalty

All five objectives are scaled to 0..1 so the weights can be compared fairly.
Constraints:
    * repair  : a deployment position or waypoint inside a no-fly zone is moved
                to the nearest open cell
    * penalty : flying THROUGH a no-fly zone, or flying further than MAX_RANGE
"""
import numpy as np



def sensor_radius(power):
    """Sensor footprint radius in cells: 1 cell at power 0, 4 cells at power 1."""
    return 1 + 3 * power


def repair(chrom, env):
    """Clip the genes to their ranges and move the deployment position and waypoints
    out of no-fly zones."""
    chrom.clip()
    for d in range(cfg.N_DRONES):
        if env.is_blocked(*chrom.deploy[d]):
            chrom.deploy[d] = env.nearest_free(*chrom.deploy[d])
        for k in range(cfg.N_WAYPOINTS):
            x, y = chrom.genes[d, k, 0], chrom.genes[d, k, 1]
            if env.is_blocked(x, y):
                chrom.genes[d, k, 0:2] = env.nearest_free(x, y)
    return chrom


def fly(chrom):
    """Turn the chromosome into a list of straight legs.

    Returns, for every leg: start point, end point, speed, sensor power,
    sensor radius (0 = sensor off) and which drone flies it (always drone 0 here).
    Leg k ends at waypoint k and uses that waypoint's speed, power and
    on/off gene. The last leg (back to the deployment position) is flown at full
    speed with the sensor off.
    """
    starts, ends, speeds, powers, radii, drone_of_leg = [], [], [], [], [], []
    for d in range(cfg.N_DRONES):
        route = chrom.route(d)
        for k in range(len(route) - 1):
            starts.append(route[k])
            ends.append(route[k + 1])
            if k < cfg.N_WAYPOINTS:
                speed, power, on = chrom.genes[d, k, 2:5]
            else:
                speed, power, on = cfg.SPEED_MAX, 0.0, 0
            speeds.append(speed)
            powers.append(power * on)                       # an OFF sensor uses no power
            radii.append(sensor_radius(power) if on else 0.0)
            drone_of_leg.append(d)
    return (np.array(starts), np.array(ends), np.array(speeds), np.array(powers),
            np.array(radii), np.array(drone_of_leg))


def distance_to_legs(px, py, starts, ends):
    """Distance from points (px, py) to every leg (a table: points x legs).

    Standard point-to-segment distance: project the point onto the leg,
    keep the projection inside the leg (0 <= t <= 1) and measure.
    """
    ax, ay = starts[:, 0][None, :], starts[:, 1][None, :]
    bx, by = ends[:, 0][None, :], ends[:, 1][None, :]
    px, py = px[:, None], py[:, None]
    abx, aby = bx - ax, by - ay
    t = np.clip(((px - ax) * abx + (py - ay) * aby) / (abx ** 2 + aby ** 2 + 1e-9), 0, 1)
    return np.hypot(px - (ax + t * abx), py - (ay + t * aby))


def coverage_map(chrom, env):
    """50x50 True/False map: True if the cell is inside a sensor footprint."""
    starts, ends, _, _, radii, _ = fly(chrom)
    dist = distance_to_legs(env.cell_x, env.cell_y, starts, ends)
    return (dist <= radii[None, :]).any(axis=1).reshape(env.size, env.size)


def points_along_legs(starts, ends):
    """Sample one point per cell flown along every leg (used for hazard / no-fly checks)."""
    xs, ys = [], []
    for a, b in zip(starts, ends):
        n = max(1, int(np.hypot(*(b - a))))
        t = np.linspace(0, 1, n + 1)[1:]
        xs.append(a[0] + t * (b[0] - a[0]))
        ys.append(a[1] + t * (b[1] - a[1]))
    return np.concatenate(xs), np.concatenate(ys)


def evaluate(chrom, env):
    """Compute all objectives and the final fitness of one chromosome."""
    starts, ends, speeds, sensor_power, radii, drone = fly(chrom)
    lengths = np.hypot(*(ends - starts).T)
    times = lengths / speeds

    # MAXIMISE 1: priority-weighted coverage of the map
    covered = coverage_map(chrom, env)
    coverage = (env.priority * covered).sum() / env.priority.sum()

    # MAXIMISE 2: share of targets that came inside a sensor footprint
    t_dist = distance_to_legs(env.targets[:, 0], env.targets[:, 1], starts, ends)
    detection = (t_dist <= radii[None, :]).any(axis=1).mean()

    # MINIMISE 1: energy. Flying faster costs more per cell; the sensor costs energy per time step.
    energy = (lengths * (1 + 0.3 * speeds) + 1.5 * sensor_power * times).sum()
    max_energy = cfg.N_DRONES * cfg.MAX_RANGE * (1 + 0.3 * cfg.SPEED_MAX + 1.5 / cfg.SPEED_MIN)
    energy_n = min(1.0, energy / max_energy)

    # MINIMISE 2: mission time = flight time of the drone
    drone_time = np.bincount(drone, times, minlength=cfg.N_DRONES)
    time_n = min(1.0, drone_time.max() / (cfg.MAX_RANGE / cfg.SPEED_MIN))

    # MINIMISE 3: hazard exposure = average hazard under the flown path
    xs, ys = points_along_legs(starts, ends)
    xi = np.clip(xs.astype(int), 0, env.size - 1)
    yi = np.clip(ys.astype(int), 0, env.size - 1)
    hazard = env.hazard[xi, yi].mean()

    # PENALTY: share of the path inside no-fly zones (x10) + flying further than allowed
    nofly = env.no_fly[xi, yi].mean()
    drone_distance = np.bincount(drone, lengths, minlength=cfg.N_DRONES)
    over_range = np.maximum(0, drone_distance / cfg.MAX_RANGE - 1).sum()
    penalty = 10 * nofly + over_range

    fitness = (cfg.W_COVERAGE * coverage + cfg.W_DETECTION * detection
               - cfg.W_ENERGY * energy_n - cfg.W_TIME * time_n - cfg.W_HAZARD * hazard
               - cfg.PENALTY_WEIGHT * penalty)

    chrom.fitness = fitness
    chrom.details = {"coverage": coverage, "detection": detection, "energy": energy_n,
                     "time": time_n, "hazard": hazard, "penalty": penalty}
    return fitness

## 5. The Genetic Algorithm (`ga.py`)

In [ ]:
"""
The Genetic Algorithm.

    1. make a random population
    2. evaluate every plan (fitness)
    3. keep the ELITE best plans
    4. fill the rest: tournament selection -> crossover -> mutation -> repair
    5. add a few random "immigrants" so the population stays diverse
    6. repeat until GENERATIONS is reached or nothing improved for PATIENCE generations
"""
import numpy as np



class GeneticAlgorithm:
    def __init__(self, env, pop_size=cfg.POP_SIZE, generations=cfg.GENERATIONS,
                 mutation_rate=cfg.MUTATION_RATE, tournament_k=cfg.TOURNAMENT_K, seed=cfg.SEED,
                 patience=cfg.PATIENCE):
        self.env = env
        self.pop_size = pop_size
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.tournament_k = tournament_k
        self.patience = patience
        self.rng = np.random.default_rng(seed)     # all randomness comes from this seed
        self.evaluations = 0                       # how many plans were evaluated
        self.history = []                          # best / average fitness per generation
        self.best_per_generation = []              # best plan of every generation (for the maps)
        self.stop_reason = ""

    # ---------- population initialisation ----------
    def new_random_plan(self):
        plan = repair(Chromosome.random(self.rng), self.env)
        evaluate(plan, self.env)
        self.evaluations += 1
        return plan

    def initial_population(self):
        return [self.new_random_plan() for _ in range(self.pop_size)]

    # ---------- selection ----------
    def tournament_selection(self, population):
        """Pick k random plans and return the best of them."""
        contestants = self.rng.choice(len(population), self.tournament_k, replace=False)
        best = max(contestants, key=lambda i: population[i].fitness)
        return population[best]

    # ---------- crossover ----------
    def crossover(self, parent1, parent2):
        """Custom two-step crossover for the drone route.

        Step 1 - one-point route crossover: the child flies the first c waypoints
                 of parent 1 and the rest of parent 2 (each part keeps its order,
                 so good pieces of route stay together).
        Step 2 - geometric blend: the deployment position and one random waypoint
                 (position, speed, power) become a random mix of both parents
                 (a point between them).
        """
        c = self.rng.integers(1, cfg.N_WAYPOINTS)                 # cut point 1..N_WAYPOINTS-1
        genes = np.concatenate([parent1.genes[:, :c], parent2.genes[:, c:]], axis=1)
        a = self.rng.random()
        deploy = a * parent1.deploy + (1 - a) * parent2.deploy
        k = self.rng.integers(cfg.N_WAYPOINTS)
        genes[:, k, 0:4] = a * parent1.genes[:, k, 0:4] + (1 - a) * parent2.genes[:, k, 0:4]
        return Chromosome(deploy, genes)

    # ---------- mutation ----------
    def mutate(self, plan):
        """Mutation designed for spatial routes.

        * move a waypoint a little (Gaussian step of about MUTATION_STEP cells)
          and change its speed and sensor power a little
        * flip a sensor on/off bit
        * move the deployment position a little
        * sometimes swap the order of two waypoints
        """
        g = plan.genes
        for d in range(cfg.N_DRONES):
            if self.rng.random() < self.mutation_rate:
                plan.deploy[d] += self.rng.normal(0, cfg.MUTATION_STEP, 2)
            for k in range(cfg.N_WAYPOINTS):
                if self.rng.random() < self.mutation_rate:
                    g[d, k, 0:2] += self.rng.normal(0, cfg.MUTATION_STEP, 2)
                    g[d, k, 2] += self.rng.normal(0, 0.3)
                    g[d, k, 3] += self.rng.normal(0, 0.2)
                if self.rng.random() < self.mutation_rate:
                    g[d, k, 4] = 1 - g[d, k, 4]
            if self.rng.random() < self.mutation_rate:
                i, j = self.rng.choice(cfg.N_WAYPOINTS, 2, replace=False)
                g[d, [i, j]] = g[d, [j, i]]
        return plan

    # ---------- main loop ----------
    def run(self):
        population = self.initial_population()
        best_ever, no_improvement = -np.inf, 0

        for generation in range(self.generations + 1):
            population.sort(key=lambda p: p.fitness, reverse=True)   # best first
            fitness = [p.fitness for p in population]
            self.history.append({"generation": generation, "best": fitness[0],
                                 "average": float(np.mean(fitness)), "evaluations": self.evaluations})
            self.best_per_generation.append(population[0].copy())

            # termination criteria
            if fitness[0] > best_ever + 1e-6:
                best_ever, no_improvement = fitness[0], 0
            else:
                no_improvement += 1
            if generation == self.generations:
                self.stop_reason = f"reached {self.generations} generations"
                break
            if no_improvement >= self.patience:
                self.stop_reason = f"no improvement for {self.patience} generations"
                break

            # next generation = elite + children + a few random immigrants
            next_population = [p.copy() for p in population[:cfg.ELITE]]
            while len(next_population) < self.pop_size - cfg.IMMIGRANTS:
                parent1 = self.tournament_selection(population)
                parent2 = self.tournament_selection(population)
                if self.rng.random() < cfg.CROSSOVER_RATE:
                    child = self.crossover(parent1, parent2)
                else:
                    child = parent1.copy()
                child = repair(self.mutate(child), self.env)
                evaluate(child, self.env)
                self.evaluations += 1
                next_population.append(child)
            while len(next_population) < self.pop_size:
                next_population.append(self.new_random_plan())
            population = next_population

        return population[0]

## 6. Baseline: random search (`random_search.py`)

In [ ]:
"""
Baseline: random search.
Make random plans (same generator and repair as the GA) and keep the best one.
It gets exactly as many evaluations as the GA, so the comparison is fair.
"""
import numpy as np



def random_search(env, n_evaluations, seed):
    rng = np.random.default_rng(seed)
    best, best_so_far = None, []
    for _ in range(n_evaluations):
        plan = repair(Chromosome.random(rng), env)
        evaluate(plan, env)
        if best is None or plan.fitness > best.fitness:
            best = plan
        best_so_far.append(best.fitness)
    return best, best_so_far

## 7. Plots (`plots.py`)

In [ ]:
"""All figures of the project (matplotlib)."""
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle


DRONE_COLOURS = ["#2a78d6", "#d55181", "#008300"]


def draw_map(ax, env):
    """Hazard (orange), no-fly zones (black), priority zones (blue boxes) and targets."""
    n = env.size
    ax.imshow(env.hazard.T, origin="lower", extent=(0, n, 0, n), cmap="Oranges", vmin=0, vmax=1.5)
    blocked = np.ma.masked_where(~env.no_fly.T, env.no_fly.T)
    ax.imshow(blocked, origin="lower", extent=(0, n, 0, n), cmap="Greys", vmin=0, vmax=1)
    for x1, y1, x2, y2, w in cfg.PRIORITY_ZONES:
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="#1f3a8a", lw=1.5, ls="--"))
    ax.scatter(env.targets[:, 0], env.targets[:, 1], marker="x", c="black", s=25, zorder=4)
    ax.set_xlim(0, n); ax.set_ylim(0, n); ax.set_aspect("equal")
    ax.set_xlabel("x (cells)"); ax.set_ylabel("y (cells)")


def draw_routes(ax, plan):
    """The drone's route: solid = sensor ON (thicker = stronger), dotted = sensor OFF.
    The triangle is the drone's deployment position."""
    for d in range(cfg.N_DRONES):
        route = plan.route(d)
        for k in range(len(route) - 1):
            if k < cfg.N_WAYPOINTS and plan.genes[d, k, 4] == 1:
                width, style = 1 + 2.5 * plan.genes[d, k, 3], "-"
            else:
                width, style = 1, ":"
            ax.plot(route[k:k + 2, 0], route[k:k + 2, 1], color=DRONE_COLOURS[d], lw=width, ls=style)
        ax.scatter(route[1:-1, 0], route[1:-1, 1], color=DRONE_COLOURS[d], s=14, zorder=3)
        ax.scatter(*plan.deploy[d], marker="^", color=DRONE_COLOURS[d], edgecolors="black", s=110, zorder=5)


def legend(ax):
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    items = [Line2D([], [], color=DRONE_COLOURS[i], lw=2, label=f"Drone {i + 1}") for i in range(cfg.N_DRONES)]
    items += [Line2D([], [], color="grey", lw=2.5, label="sensor ON (thicker = stronger)"),
              Line2D([], [], color="grey", ls=":", label="sensor OFF"),
              Patch(color="#555555", label="no-fly zone"), Patch(color="#f4a261", label="hazard zone"),
              Patch(fill=False, ec="#1f3a8a", ls="--", label="priority zone"),
              Line2D([], [], marker="x", ls="", color="black", label="target"),
              Line2D([], [], marker="^", ls="", color="grey", mec="black", label="deployment position")]
    ax.legend(handles=items, loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)


def title_of(plan):
    d = plan.details
    return (f"F={plan.fitness:.3f}  cov={d['coverage']:.2f}  det={d['detection']:.2f}\n"
            f"energy={d['energy']:.2f}  time={d['time']:.2f}  hazard={d['hazard']:.2f}  penalty={d['penalty']:.2f}")


def plot_convergence(history, path):
    gens = [h["generation"] for h in history]
    plt.figure(figsize=(7, 4))
    plt.plot(gens, [h["best"] for h in history], label="best fitness", lw=2)
    plt.plot(gens, [h["average"] for h in history], label="average fitness", lw=2)
    plt.xlabel("generation"); plt.ylabel("fitness"); plt.title("Convergence: best vs. average fitness")
    plt.grid(alpha=0.3); plt.legend()
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


def plot_paths_over_generations(ga, env, gens, path):
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    for ax, g in zip(axes.ravel(), gens):
        plan = ga.best_per_generation[g]
        draw_map(ax, env); draw_routes(ax, plan)
        ax.set_title(f"Generation {g}\n{title_of(plan)}", fontsize=8)
    legend(axes[0, 1])
    fig.suptitle("Best flight paths across generations")
    fig.tight_layout(); fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)


def plot_coverage_over_generations(ga, env, gens, path):
    fig, axes = plt.subplots(1, len(gens), figsize=(4 * len(gens), 4.2))
    for ax, g in zip(axes, gens):
        plan = ga.best_per_generation[g]
        seen = np.where(coverage_map(plan, env), env.priority, np.nan)   # NaN = not covered (white)
        im = ax.imshow(seen.T, origin="lower", extent=(0, env.size, 0, env.size), cmap="viridis", vmin=0, vmax=1)
        blocked = np.ma.masked_where(~env.no_fly.T, env.no_fly.T)
        ax.imshow(blocked, origin="lower", extent=(0, env.size, 0, env.size), cmap="Greys", vmin=0, vmax=1)
        ax.set_title(f"Generation {g}: coverage {plan.details['coverage']:.2f}", fontsize=9)
        ax.set_xlabel("x (cells)")
    axes[0].set_ylabel("y (cells)")
    fig.colorbar(im, ax=list(axes), label="priority of covered cell (white = not covered)", shrink=0.8)
    fig.suptitle("Coverage heatmaps across generations")
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)


def plot_sensitivity(curves, path):
    """curves = {sweep name: {setting label: [best-fitness curve of each seed]}}"""
    fig, axes = plt.subplots(1, len(curves), figsize=(5 * len(curves), 4))
    for ax, (name, settings) in zip(axes, curves.items()):
        for label, runs in settings.items():
            mean = np.mean(runs, axis=0)
            ax.plot(mean, lw=2, label=label)
        ax.set_title(name); ax.set_xlabel("generation"); ax.set_ylabel("best fitness (mean of seeds)")
        ax.grid(alpha=0.3); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def plot_ga_vs_random(ga_curves, rs_curves, path):
    plt.figure(figsize=(7, 4))
    for label, curves in (("Genetic Algorithm", ga_curves), ("Random search", rs_curves)):
        x = curves[0][0]
        mean = np.mean([c[1] for c in curves], axis=0)
        plt.plot(x, mean, lw=2, label=f"{label} (mean of {len(curves)} seeds)")
    plt.xlabel("number of fitness evaluations"); plt.ylabel("best fitness found so far")
    plt.title("GA vs. random search (same number of evaluations)")
    plt.grid(alpha=0.3); plt.legend()
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


def plot_two_plans(ga_plan, rs_plan, env, path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    for ax, plan, name in ((axes[0], ga_plan, "Genetic Algorithm"), (axes[1], rs_plan, "Random search")):
        draw_map(ax, env); draw_routes(ax, plan)
        ax.set_title(f"{name}\n{title_of(plan)}", fontsize=9)
    legend(axes[1])
    fig.tight_layout(); fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)

## 8. Experiments (`main.py`)

In [ ]:
"""
Runs everything and saves the results in the results/ folder.

    In Colab: Runtime -> Run all   (all experiments take about 5 minutes)
"""
import csv
import os

import numpy as np



def save_csv(rows, path):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)


def experiment_main(env, out, generations):
    """One GA run with the default settings + all its figures."""
    print("1) Main GA run")
    ga = GeneticAlgorithm(env, generations=generations)
    best = ga.run()
    last = len(ga.history) - 1
    gens = [0, last // 4, last // 2, last]                 # 4 snapshots for the maps
    save_csv(ga.history, f"{out}/history.csv")
    plot_convergence(ga.history, f"{out}/convergence.png")
    plot_paths_over_generations(ga, env, gens, f"{out}/paths_generations.png")
    plot_coverage_over_generations(ga, env, gens, f"{out}/coverage_heatmaps.png")
    summary = {"stop_reason": ga.stop_reason, "generations_run": last, "evaluations": ga.evaluations,
               "first_best": ga.history[0]["best"], "first_average": ga.history[0]["average"],
               "final_best": best.fitness, "final_average": ga.history[-1]["average"], **best.details}
    save_csv([summary], f"{out}/main_summary.csv")
    print(f"   best fitness {ga.history[0]['best']:.3f} -> {best.fitness:.3f} ({ga.stop_reason})")


def experiment_sensitivity(env, out, generations, seeds):
    """Change ONE setting at a time and see how the final best fitness changes."""
    print("2) Sensitivity analysis")
    sweeps = {"Population size": ("pop_size", [20, 50, 100]),
              "Mutation rate": ("mutation_rate", [0.02, 0.1, 0.3]),
              "Selection pressure (tournament k)": ("tournament_k", [2, 3, 5])}
    rows, curves = [], {}
    for name, (parameter, values) in sweeps.items():
        curves[name] = {}
        for value in values:
            finals, runs = [], []
            for seed in seeds:
                ga = GeneticAlgorithm(env, generations=generations, seed=seed,
                                      patience=generations, **{parameter: value})
                best = ga.run()
                finals.append(best.fitness)
                runs.append([h["best"] for h in ga.history])
            curves[name][f"{parameter} = {value}"] = runs
            rows.append({"parameter": parameter, "value": value, "mean_best": np.mean(finals),
                         "std_best": np.std(finals), "evaluations": ga.evaluations})
            print(f"   {parameter} = {value}: {np.mean(finals):.3f} +/- {np.std(finals):.3f}")
    save_csv(rows, f"{out}/sensitivity.csv")
    plot_sensitivity(curves, f"{out}/sensitivity.png")


def experiment_baseline(env, out, generations, seeds):
    """GA vs. random search with exactly the same number of evaluations."""
    print("3) GA vs. random search")
    rows, ga_curves, rs_curves = [], [], []
    for seed in seeds:
        ga = GeneticAlgorithm(env, generations=generations, seed=seed)
        ga_best = ga.run()
        rs_best, rs_curve = random_search(env, ga.evaluations, seed)
        evals = [h["evaluations"] for h in ga.history]
        ga_curves.append((evals, [h["best"] for h in ga.history]))
        rs_curves.append((evals, [rs_curve[e - 1] for e in evals]))
        for method, plan in (("GA", ga_best), ("Random search", rs_best)):
            rows.append({"method": method, "seed": seed, "fitness": plan.fitness,
                         "evaluations": ga.evaluations, **plan.details})
        print(f"   seed {seed}: GA {ga_best.fitness:.3f}  random {rs_best.fitness:.3f}")
        if seed == seeds[0]:
            plot_two_plans(ga_best, rs_best, env, f"{out}/ga_vs_random_maps.png")
    save_csv(rows, f"{out}/baseline.csv")
    plot_ga_vs_random(ga_curves, rs_curves, f"{out}/ga_vs_random.png")
    for method in ("GA", "Random search"):
        values = [r["fitness"] for r in rows if r["method"] == method]
        print(f"   {method}: {np.mean(values):.3f} +/- {np.std(values):.3f}")

## 9. Experiment 1 – Main GA run (about 20 seconds)

In [ ]:
from IPython.display import Image, display
out = "results"
os.makedirs(out, exist_ok=True)
env = Environment()
experiment_main(env, out, cfg.GENERATIONS)
for name in ["convergence", "paths_generations", "coverage_heatmaps"]:
    display(Image(f"{out}/{name}.png", width=800))

## 10. Experiment 2 – Hyperparameter sensitivity (about 3 minutes)

In [ ]:
experiment_sensitivity(env, out, cfg.GENERATIONS, seeds=[1, 2, 3, 4, 5])
display(Image(f"{out}/sensitivity.png", width=900))

## 11. Experiment 3 – GA vs. random search (about 1 minute)

In [ ]:
experiment_baseline(env, out, cfg.GENERATIONS, seeds=[1, 2, 3, 4, 5])
display(Image(f"{out}/ga_vs_random.png", width=700))
display(Image(f"{out}/ga_vs_random_maps.png", width=900))